# IU-CXR Preprocessing — Kaggle Notebook

> 🎯 **Mục đích**: Process raddar's `chest-xrays-indiana-university` Kaggle Dataset → ra format Forget-MI compatible. Output thành Kaggle Dataset `forget-mi-data-iu` để dùng cho training notebooks.

## Workflow

1. **Add Input** (Sidebar → + Add data): search `raddar chest xrays indiana university` → Add
2. **Run all cells** (~2-3 phút)
3. **Save Version** → Save & Run All
4. **Sau khi Successful** → click "Create Dataset from Output" → name `forget-mi-data-iu`
5. Dùng dataset đó trong `run_kaggle_baseline.ipynb` Cell 4d/4e/4f và `run_kaggle_loku.ipynb`

## Output structure

```
/kaggle/working/data_iu/
├── data/
│   ├── metadata/
│   │   └── all_data.tsv        ← consumed by Forget-MI processor
│   └── img_data/               ← symlinks to raddar PNGs (no copy)
└── data_splits/
    ├── iu-split.csv
    ├── forget_set_3per_iu.csv
    ├── forget_set_6per_iu.csv
    └── forget_set_10per_iu.csv
```

In [ ]:
# CELL 1: Setup — clone repo + verify input dataset
import os, subprocess, glob

WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/Forget-MI-LoKU"

# Token-auth clone (repo có thể private)
def _get_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('GITHUB_TOKEN')
    except Exception:
        return None

token = _get_token()
url = f"https://{token}@github.com/nhnhu146/Forget-MI-LoKU.git" if token else "https://github.com/nhnhu146/Forget-MI-LoKU.git"

os.chdir(WORK)
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', url, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/master'], check=True)
os.chdir(REPO_DIR)
print(f"📂 Repo at: {os.getcwd()}")
subprocess.run(['git', 'log', '--oneline', '-1'])

# Auto-detect raddar dataset
def _find_dataset(slug):
    direct = f'/kaggle/input/{slug}'
    if os.path.isdir(direct): return direct
    candidates = glob.glob(f'/kaggle/input/datasets/*/{slug}')
    return candidates[0] if candidates else None

# raddar's slug variants
IU_ROOT = (_find_dataset('chest-xrays-indiana-university')
           or _find_dataset('iu-xray-dataset')
           or _find_dataset('indiana-university-chest-xrays'))

if not IU_ROOT:
    print("\n❌ KHÔNG tìm thấy raddar's IU dataset trong /kaggle/input/")
    print("   → Sidebar → + Add data → search 'raddar chest xrays indiana university'")
    raise SystemExit
print(f"\n📦 IU dataset root: {IU_ROOT}")
print(f"   Contents: {os.listdir(IU_ROOT)[:10]}")

# Verify expected files
for f in ['indiana_reports.csv', 'indiana_projections.csv']:
    path = os.path.join(IU_ROOT, f)
    print(f"  {'✅' if os.path.exists(path) else '❌'} {f}")

# Find images directory (might be 'images/images' or 'images_normalized')
img_dir = None
for candidate in ['images/images', 'images_normalized', 'images']:
    p = os.path.join(IU_ROOT, candidate)
    if os.path.isdir(p):
        # Check has PNGs
        if glob.glob(os.path.join(p, '*.png'))[:1]:
            img_dir = p
            break
print(f"\n🖼  Images dir: {img_dir}")
if img_dir:
    n = len(glob.glob(os.path.join(img_dir, '*.png')))
    print(f"   PNG count: {n}")


In [ ]:
# CELL 2: Parse reports CSV → JSON
import subprocess

OUT_JSON = "/kaggle/working/data_iu/parsed/reports.json"
cmd = [
    'python', 'scripts/parse_iu_kaggle.py',
    '--reports_csv', f'{IU_ROOT}/indiana_reports.csv',
    '--projections_csv', f'{IU_ROOT}/indiana_projections.csv',
    '--output', OUT_JSON,
]
if img_dir:
    cmd += ['--img_dir', img_dir]

print(' '.join(cmd))
result = subprocess.run(cmd, capture_output=False, text=True)
if result.returncode != 0:
    raise RuntimeError(f"parse failed (exit {result.returncode})")


In [ ]:
# CELL 3: Build all_data.tsv + splits + (symlink) images
import subprocess

OUT_DIR = "/kaggle/working/data_iu"
cmd = [
    'python', 'scripts/make_iu_dataset.py',
    '--reports', f'{OUT_DIR}/parsed/reports.json',
    '--img_src', img_dir,
    '--out_dir', OUT_DIR,
    '--link_mode', 'symlink',   # symlink saves disk on Kaggle
]
print(' '.join(cmd))
result = subprocess.run(cmd, capture_output=False, text=True)
if result.returncode != 0:
    raise RuntimeError(f"make_iu_dataset failed (exit {result.returncode})")


In [ ]:
# CELL 4: Verify output + show stats
import os, glob

OUT_DIR = "/kaggle/working/data_iu"

# Files generated
expected = [
    f"{OUT_DIR}/data/metadata/all_data.tsv",
    f"{OUT_DIR}/data_splits/iu-split.csv",
    f"{OUT_DIR}/data_splits/forget_set_3per_iu.csv",
    f"{OUT_DIR}/data_splits/forget_set_6per_iu.csv",
    f"{OUT_DIR}/data_splits/forget_set_10per_iu.csv",
]
print("📋 Generated files:")
for p in expected:
    ok = os.path.exists(p)
    size = os.path.getsize(p) if ok else 0
    print(f"  {'✅' if ok else '❌'} {p}  ({size} bytes)")

# Show all_data.tsv head
print("\n📄 all_data.tsv (first 3 lines):")
with open(f"{OUT_DIR}/data/metadata/all_data.tsv") as f:
    for i, line in enumerate(f):
        if i >= 3: break
        print(f"  {line.rstrip()[:200]}")

# Show split CSV head
print("\n📄 iu-split.csv (first 5 rows):")
import pandas as pd
df = pd.read_csv(f"{OUT_DIR}/data_splits/iu-split.csv")
print(df.head().to_string())
print(f"\nLabel distribution:")
print(df['edeme_severity'].value_counts())
print(f"\nFold distribution:")
print(df['fold'].value_counts())

# Image symlinks
n_imgs = len(glob.glob(f"{OUT_DIR}/data/img_data/*.png"))
print(f"\n🖼  Image symlinks: {n_imgs}")

# Forget sets
print("\n📋 Forget sets:")
for pct in [3, 6, 10]:
    fpath = f"{OUT_DIR}/data_splits/forget_set_{pct}per_iu.csv"
    fdf = pd.read_csv(fpath)
    print(f"  {pct}%: {len(fdf)} subjects")


## ✅ Sau khi notebook chạy xong

1. **File → Save Version → Save & Run All**
2. Đợi notebook xong (~3-5 phút)
3. Sau khi Successful → click button **"Create Dataset from Output"** (trên trang notebook)
4. Đặt name: `forget-mi-data-iu` (lowercase, hyphen)
5. Visibility: Private (hoặc Public tùy bạn)
6. Click "Create"

→ Dataset mới sẽ available cho mọi notebook của bạn.

## ⚠️ Lưu ý quan trọng về symlink

Notebook dùng `--link_mode symlink` để tiết kiệm disk. NHƯNG khi tạo Kaggle Dataset từ output, **Kaggle sẽ resolve symlinks và copy file thật**. → Dataset cuối sẽ có ~2 GB images (đầy đủ, không phải symlink).

Nếu muốn dataset NHỎ HƠN (chỉ metadata + splits, KHÔNG images), set `--link_mode skip` trong Cell 3. Sau đó training notebook sẽ trỏ ảnh tới raddar's dataset trực tiếp (cần thêm `forget-mi-data-iu` + `chest-xrays-indiana-university` cùng làm Input).

## 🚀 Next: tạo training notebook

Sau khi `forget-mi-data-iu` dataset ready, dùng `run_kaggle_baseline.ipynb` Cell 4d/4e/4f (IU cells) với `RUN_IU_3PER=True`. Hoặc tạo notebook chuyên IU.

Tôi sẽ viết tiếp `train_iu_original_model.py` (cho model_og + model_re) — bạn báo khi notebook này xong.